# SQL指令對話機器人

# 選擇模型
請自由任意選擇下面兩個模型之一來進行範例  

## Gemini

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

API_KEY = "這邊請改成你自己的API_KEY值"
model_name = 'gemini-2.5-flash'

llm = ChatGoogleGenerativeAI(
    model=model_name,
    google_api_key=API_KEY
)

## LM Studio 或 Ollama

In [ ]:
from langchain_openai import ChatOpenAI
model_name = 'gemma-3-12b-it'  # 指定模型名稱，模型名稱會根據下載的模型不同而改變

# 根據使用LM Studio或Ollama來選擇適當的伺服器URL
base_url = 'http://localhost:1234/v1'  # LM Studio 本地伺服器的URL
# base_url = 'http://localhost:11434/v1' # Ollama 本地伺服器的URL

llm = ChatOpenAI(
    model=model_name,
    openai_api_key="not-needed",
    openai_api_base=base_url 
)

## 測試模型

In [4]:
from langchain_core.messages import HumanMessage
messages = [
    HumanMessage("機器學習的定義")
]
result = llm.invoke(messages)
print(result.content)

機器學習（Machine Learning, ML）是人工智慧（Artificial Intelligence, AI）的一個子領域，它賦予電腦系統從資料中「學習」的能力，而無需被明確地編程來執行每一步任務。

更具體地說，機器學習的核心定義和目標是：

1.  **從資料中學習：** 機器學習演算法透過分析大量的資料（例如，圖片、文本、數字等），自動識別出資料中的模式、趨勢和關聯性。
2.  **無需明確編程：** 與傳統編程不同，傳統編程需要開發者為每個可能的情況編寫詳細的規則。機器學習則允許系統透過經驗（資料）自動發現這些規則和模式。
3.  **做出預測或決策：** 基於從資料中學習到的模式，機器學習模型能夠對新的、未見過的資料做出預測（例如，預測股價、天氣），或執行特定的任務（例如，分類圖片、識別語音）。
4.  **性能隨經驗提升：** 機器學習系統的性能會隨著時間的推移和更多資料的輸入而提高。它會不斷調整其內部參數，以優化其在特定任務上的表現。

**總之，機器學習是讓電腦透過經驗（資料）來學習和改進的科學，是現代人工智慧發展的核心驅動力之一。**

**主要特點：**

*   **資料驅動：** 資料是機器學習的燃料。
*   **模式識別：** 核心是從資料中找出有意義的模式。
*   **自動化學習：** 減少人工編寫複雜規則的需求。
*   **泛化能力：** 學習到的模型能夠應用於新的、未見過的資料。

**常見的機器學習類型包括：**

*   **監督式學習（Supervised Learning）：** 資料帶有標籤，模型學習輸入與輸出之間的映射關係（如分類、迴歸）。
*   **非監督式學習（Unsupervised Learning）：** 資料不帶標籤，模型尋找資料內部的結構或模式（如聚類、降維）。
*   **強化學習（Reinforcement Learning）：** 代理（Agent）透過與環境互動，根據獎勵和懲罰來學習最佳行為策略。

機器學習的應用無處不在，從語音識別、圖像識別、推薦系統到自動駕駛和醫療診斷等。


# SQL指令對話機器人

In [2]:
# 資料庫 Schema
schema = '''
-- ===============================
-- 顧客購買紀錄資料庫 Schema
-- ===============================

-- 1. 區域資料
CREATE TABLE Regions (
    RegionID    INT PRIMARY KEY AUTO_INCREMENT,
    RegionName  VARCHAR(100) NOT NULL UNIQUE
);

-- 2. 職業類別資料
CREATE TABLE Occupations (
    OccupationID   INT PRIMARY KEY AUTO_INCREMENT,
    OccupationName VARCHAR(100) NOT NULL UNIQUE
);

-- 3. 顧客資料
CREATE TABLE Customers (
    CustomerID      INT PRIMARY KEY AUTO_INCREMENT,
    Name            VARCHAR(100) NOT NULL,
    BirthDate       DATE,
    RegionID        INT,
    OccupationID    INT,
    CONSTRAINT fk_customer_region FOREIGN KEY (RegionID) REFERENCES Regions(RegionID),
    CONSTRAINT fk_customer_occupation FOREIGN KEY (OccupationID) REFERENCES Occupations(OccupationID)
);

CREATE INDEX idx_customer_region ON Customers(RegionID);
CREATE INDEX idx_customer_occupation ON Customers(OccupationID);

-- 4. 分店資料
CREATE TABLE Stores (
    StoreID     INT PRIMARY KEY AUTO_INCREMENT,
    RegionID    INT,
    StoreName   VARCHAR(100) NOT NULL,
    FoundedDate DATE,
    CONSTRAINT fk_store_region FOREIGN KEY (RegionID) REFERENCES Regions(RegionID)
);

CREATE INDEX idx_store_region ON Stores(RegionID);

-- 5. 產品資料
CREATE TABLE Products (
    ProductID   INT PRIMARY KEY AUTO_INCREMENT,
    ProductName VARCHAR(100) NOT NULL,
    Category    VARCHAR(50),
    UnitPrice   DECIMAL(10,2) NOT NULL
);

CREATE INDEX idx_product_category ON Products(Category);

-- 6. 銷售紀錄
CREATE TABLE Sales (
    SaleID      BIGINT PRIMARY KEY AUTO_INCREMENT,   -- 交易序號
    CustomerID  INT NOT NULL,
    StoreID     INT NOT NULL,
    TotalAmount DECIMAL(12,2) NOT NULL,
    SaleDate    DATETIME NOT NULL DEFAULT CURRENT_TIMESTAMP,
    CONSTRAINT fk_sales_customer FOREIGN KEY (CustomerID) REFERENCES Customers(CustomerID),
    CONSTRAINT fk_sales_store FOREIGN KEY (StoreID) REFERENCES Stores(StoreID)
);

CREATE INDEX idx_sales_customer ON Sales(CustomerID);
CREATE INDEX idx_sales_store ON Sales(StoreID);
CREATE INDEX idx_sales_date ON Sales(SaleDate);

-- 7. 交易細項紀錄
CREATE TABLE SaleItems (
    SaleID     BIGINT,
    ProductID  INT,
    Quantity   INT NOT NULL,
    UnitPrice  DECIMAL(10,2) NOT NULL,
    PRIMARY KEY (SaleID, ProductID),
    CONSTRAINT fk_saleitems_sale FOREIGN KEY (SaleID) REFERENCES Sales(SaleID),
    CONSTRAINT fk_saleitems_product FOREIGN KEY (ProductID) REFERENCES Products(ProductID)
);

CREATE INDEX idx_saleitems_product ON SaleItems(ProductID);
'''

In [3]:
system_prompt = f"""
你是一位專業的資料庫管理員（DBA），負責根據使用者的自然語言問題，產生可在資料庫中查詢出相關資料的 **SQL 指令**。

請根據以下的資料庫結構（Schema）進行理解，並依據使用者問題生成正確、可執行且語法完整的 SQL 查詢語句。

【資料庫結構說明】
{schema}
【說明結束】

請注意以下原則：
1. 僅產生與查詢（SELECT）有關的 SQL 指令，不要進行資料新增、修改或刪除（INSERT、UPDATE、DELETE）。
2. 不要虛構不存在的資料表或欄位。
3. 若使用者的問題模糊，請根據 schema 中的合理欄位推斷最可能的查詢內容。
4. 僅輸出 SQL 指令，不要包含任何解釋或自然語言說明。
5. 保持語法正確，並依標準 SQL 語法格式化。

範例：
使用者問題：「查詢上個月每個地區的銷售總額」
輸出：
SELECT region, SUM(sales_amount)
FROM sales
WHERE month = '2025-09'
GROUP BY region;

現在開始等待使用者的查詢問題。
"""


In [4]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
class SQL_Bot:
    def __init__(self, llm):
        self.llm = llm  # LLM 是對話機器人的大腦
        
        # 初始化對話記錄，包含系統提示詞
        self.messages = [SystemMessage(system_prompt)]

    # 一般版：一次性呼叫模型，整段生成完才回傳
    def chat(self, text):
        # 將使用者輸入加入記錄
        self.messages.append(HumanMessage(text))
        # 呼叫模型生成完整回覆
        response = self.llm.invoke(self.messages)
        # 將模型回覆加入記錄
        self.messages.append(response)
        # 回傳文字內容
        return response.content

    # 串流版：使用 yield 逐步回傳生成結果
    def chat_stream(self, text):
        # 將使用者輸入加入記錄
        self.messages.append(HumanMessage(text))
        
        # 用於累積完整回覆
        full_response = ""

        # 使用 llm.stream() 逐步獲取生成內容
        for chunk in self.llm.stream(self.messages):
            # chunk.content 為每次生成的一小段文字
            if chunk.content:
                full_response += chunk.content
                # 使用 yield 將部分內容回傳給外部呼叫者
                yield chunk.content

        # 串流結束後，將完整回覆加入記錄
        self.messages.append(AIMessage(full_response))

In [6]:
bot = SQL_Bot(llm)
print(bot.chat("各個區域的顧客總共花了多少錢？哪個區域的消費最多？"))

```sql
SELECT
  R.RegionName,
  SUM(S.TotalAmount) AS TotalSpent
FROM Regions AS R
JOIN Customers AS C
  ON R.RegionID = C.RegionID
JOIN Sales AS S
  ON C.CustomerID = S.CustomerID
GROUP BY
  R.RegionName
ORDER BY
  TotalSpent DESC;
```


In [7]:
bot = SQL_Bot(llm)
print(bot.chat("哪些產品最暢銷？每個產品賣出了多少件？"))

```sql
SELECT
  p.ProductName,
  SUM(si.Quantity) AS TotalQuantitySold
FROM SaleItems AS si
JOIN Products AS p
  ON si.ProductID = p.ProductID
GROUP BY
  p.ProductID,
  p.ProductName
ORDER BY
  TotalQuantitySold DESC;
```


In [8]:
bot = SQL_Bot(llm)
print(bot.chat("不同職業的顧客平均每次消費金額是多少？哪個職業的顧客消費能力最高？"))

```sql
SELECT
  o.OccupationName,
  AVG(s.TotalAmount) AS AverageSpendingPerSale
FROM Sales AS s
JOIN Customers AS c
  ON s.CustomerID = c.CustomerID
JOIN Occupations AS o
  ON c.OccupationID = o.OccupationID
GROUP BY
  o.OccupationName
ORDER BY
  AverageSpendingPerSale DESC;
```


# 與SQL指令對話機器人交談的介面

In [9]:
# 載入 Gradio 套件，用於建立網頁互動介面
import gradio as gr  

# ChatBot 是先前定義的聊天機器人類別
# 以 llm (大型語言模型) 為參數建立一個機器人實例
bot = SQL_Bot(llm)  

# 定義主要的對話處理函式，支援串流輸出
def chat_function_stream(message, history):
    """
    處理每輪使用者輸入的訊息，並以串流方式回傳回應。
    
    參數：
        message (str)：使用者輸入的文字
        history (list)：對話歷史紀錄，每輪對話包含問答內容
    
    回傳：
        生成器 (generator)：逐步產生回覆的文字片段，可即時在 Gradio 顯示
    """
    # 用於累積完整回覆
    full_response = ""
    
    # 使用 chat_stream 生成器逐步取得模型輸出
    for chunk in bot.chat_stream(message):
        # 將每次生成的片段累加
        full_response += chunk
        # 使用 yield 將目前累積的回覆回傳給 Gradio
        yield full_response

# 建立 Gradio 聊天介面
# - chat_function_stream：每次使用者輸入時呼叫的函式
webui = gr.ChatInterface(chat_function_stream)

# 啟動 Gradio 網頁伺服器
# 使用者可透過瀏覽器訪問本機介面 (例如 http://127.0.0.1:7860)
webui.launch()


* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.
